# Hermite Order Comparison

This notebook compares adaptive Hermite-4, Hermite-6, and Hermite-8 on the same two-body orbit.
It reports accepted step counts, final energy drift, and angular-momentum drift.

In [ ]:
import jax
import jax.numpy as jnp

from nornax import (
    AarsethController,
    initialize_state,
    solve_adaptive_to_time,
    total_angular_momentum,
    total_energy,
)
from nornax.forces import DirectSumGravity

jax.config.update("jax_enable_x64", True)

In [ ]:
force_model = DirectSumGravity()
positions = jnp.asarray([[-1.0, 0.0, 0.0], [1.0, 0.0, 0.0]])
velocities = jnp.asarray([[0.0, 0.5, 0.0], [0.0, -0.5, 0.0]])
masses = jnp.asarray([1.0, 1.0])
reference = initialize_state(positions, velocities, masses, force_model, max_order=4)
e0 = float(total_energy(reference))
l0 = total_angular_momentum(reference)

controllers = {
    4: AarsethController(eta=0.02, min_dt=1.0e-4, max_dt=5.0e-2),
    6: AarsethController(eta=0.04, min_dt=1.0e-4, max_dt=5.0e-2),
    8: AarsethController(eta=0.08, min_dt=1.0e-4, max_dt=5.0e-2),
}

In [ ]:
rows = []
for order in (4, 6, 8):
    result = solve_adaptive_to_time(
        positions,
        velocities,
        masses,
        force_model,
        t_final=2.0,
        order=order,
        controller=controllers[order],
        atol=1.0e-8,
    )
    ef = float(total_energy(result.final_state))
    lf = total_angular_momentum(result.final_state)
    rows.append(
        {
            "order": order,
            "accepted_steps": int(result.dt_history.shape[0]),
            "energy_drift": abs(ef - e0),
            "angular_momentum_drift": float(jnp.linalg.norm(lf - l0)),
        }
    )
rows